# SMS backup cleanup — merge everything into one file

Run this **signed in to the Google account that owns the backups**. Colab runs in Google's cloud and reads your Drive directly; nothing is downloaded to your computer.

Run the cells **in order, top to bottom**. Each cell's title says which step it is, and the comments at the top of each cell say what it does, what to do, and what to look for before moving on.

| Step | What | Time |
|---|---|---|
| **1** | Mount Google Drive | seconds |
| **2** | Get the cleanup tool | seconds |
| **3** | Point to your backup folder(s) and check the list of backups | seconds |
| *3b* | *Optional:* find every folder in My Drive that holds backups | 1–2 min |
| **4** | *Only if you have an older `…-cleanup` folder:* verify it, then delete its `superseded` folder | ~1 h per 125 GB |
| **5** | Merge every backup into **one** file | ~1 h per 125 GB |
| **6** | Verify the merged file; originals are moved to `…-to-delete` | ~1 h per 125 GB |
| **7** | Delete `…-to-delete`, empty the Drive trash | minutes |

In [ ]:
# @title Step 1 — Mount Google Drive
# WHAT: Gives this notebook access to your Google Drive.
# DO:   Run this cell. In the popup, choose the Google account that OWNS the
#       backups (the one whose Drive holds the sms-*.xml files) and allow access.
# NEXT: When it prints "Mounted at /content/drive", go to Step 2.
#       Wrong account? Run:  drive.mount('/content/drive', force_remount=True)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# @title Step 2 — Get the cleanup tool
# WHAT: Downloads the latest version of the tool from GitHub. Safe to re-run.
# DO:   Run this cell.
# NEXT: When it prints the latest change (a short code and a description), go to Step 3.
%cd /content
!rm -rf /content/SMS-Cleanup && git clone -q https://github.com/LegitUnicorn/SMS-Cleanup.git /content/SMS-Cleanup
%cd /content/SMS-Cleanup
!git --no-pager log --oneline --no-decorate --no-color -1

In [ ]:
# @title Step 3 — Point to your backup folder(s)
# WHAT: Tells the tool where your backups are.
#       BACKUP_DIR    = the main folder (the one the backup app saves to). The merged
#                       file is written next to it, in "<BACKUP_DIR>-merged".
#       OTHER_FOLDERS = any OTHER folders with backups to clean up too: an older phone's
#                       backups, copies, a folder of .zip backups, ... Separate several
#                       with ;   Leave empty if there are none. (Step 3b can find them.)
# DO:   Fill in the boxes on the right (in Colab), each as a path such as
#         /content/drive/MyDrive/SMS          for a folder "SMS" in My Drive
#       or as a folder link copied from drive.google.com. Then run this cell.
# NEXT: It lists EVERY backup Steps 5 and 6 will read -- subfolders, call logs and
#       backups inside .zip files included -- grouped by folder. Check the list:
#       everything listed is merged into one file and afterwards moved to "-to-delete".
#       If something should NOT be included, move it out of those folders and re-run.
#       Duplicates, incremental backups and overlapping copies are fine: every message
#       is kept exactly once, and the merge report shows what each file contributed.
#       Then: if it mentions an older "-cleanup" folder, do Step 4; otherwise skip to Step 5.
import os, shlex

BACKUP_DIR = '/content/drive/MyDrive/SMSBackupRestore'  # @param {type:"string"}
OTHER_FOLDERS = ''  # @param {type:"string"}

def _resolve(p):
    p = p.strip()
    if p.startswith('http'):
        from smscleanup.colab import resolve_folder_link
        p = resolve_folder_link(p)  # also checks you own the folder
    return p.rstrip('/')

BACKUP_DIR    = _resolve(BACKUP_DIR)
OTHERS        = [_resolve(p) for p in OTHER_FOLDERS.split(';') if p.strip()]
MERGED_DIR    = BACKUP_DIR + '-merged'      # the merged file(s) end up here (KEEP this folder)
TO_DELETE_DIR = BACKUP_DIR + '-to-delete'   # verified originals are moved here (delete this folder)
OLD_WORK_DIR  = BACKUP_DIR + '-cleanup'     # only exists if you used the older analyze/apply steps

from smscleanup.sources import collect_inputs, describe_sources, find_backup_folders

bad = [p for p in OTHERS if not os.path.isdir(p)]
for p in bad:
    print(f'!!! OTHER_FOLDERS: "{p}" is not a folder -- fix it and run this cell again.')
INPUTS = [BACKUP_DIR] + [p for p in OTHERS if p not in bad] + ([OLD_WORK_DIR] if os.path.isdir(OLD_WORK_DIR) else [])
INPUT_ARGS = ' '.join(shlex.quote(p) for p in INPUTS)

skipped = []
sources = collect_inputs(INPUTS, exclude=[MERGED_DIR, TO_DELETE_DIR], problems=skipped) if os.path.isdir(BACKUP_DIR) else []
if not sources:
    print(f'No sms-/calls- backups found in {BACKUP_DIR}')
    print('Searching your whole Drive for them (can take a minute)...')
    hits = find_backup_folders('/content/drive/MyDrive')
    for d, c in sorted(hits.items()):
        print(f'  {d}  ({", ".join(f"{n} .{k}" for k, n in sorted(c.items()))})')
    if hits:
        print('\n>>> Put the main folder in BACKUP_DIR (any others in OTHER_FOLDERS) and run this cell again.')
    else:
        print('Nothing found under My Drive. If the backups are in "Shared with me", '
              'add a shortcut to that folder in My Drive first.')
else:
    print('Steps 5 and 6 will read ALL of these (subfolders and .zip files included):')
    print('\n'.join(describe_sources(sources)))
    if skipped:
        print('\nThese will be SKIPPED and left where they are:')
        for p in skipped:
            print('   ', p)
    print('\nMerged file(s) will be written to:', MERGED_DIR)
    if os.path.isdir(os.path.join(OLD_WORK_DIR, 'superseded')):
        print(f'Found an older cleanup folder ({OLD_WORK_DIR}) -> do Step 4 next.')
    else:
        print('Next: skip Step 4 and go to Step 5.' + ('' if OTHERS else
              '  (Backups in other folders too? Run Step 3b first.)'))

In [ ]:
# @title Step 3b (optional) — Find every folder in My Drive that holds backups
# WHAT: Looks through your whole My Drive for folders with sms-/calls- backups
#       (.xml files, or .zip files named sms-... / calls-...). Takes a minute or two.
# DO:   Run it if backups might be in more than one place (older phones, copies,
#       a folder of .zip files).
# NEXT: Folders marked "NOT included" are not part of the cleanup yet. To include one,
#       add its path to OTHER_FOLDERS in Step 3 (separate several with ;), run Step 3
#       again and check its list. Then continue with Step 4 or Step 5.
#       (Zips with other names aren't spotted here, but once their folder is added,
#       Step 3 looks inside every .zip in it.)
included = [os.path.realpath(p) for p in INPUTS]
hits = find_backup_folders('/content/drive/MyDrive')
if not hits:
    print('No folders with sms-/calls- backups found under My Drive.')
for d, c in sorted(hits.items()):
    real = os.path.realpath(d)
    inc = any(real == p or real.startswith(p + os.sep) for p in included)
    what = ', '.join(f'{n} .{k}' for k, n in sorted(c.items()))
    print(f'{"included    " if inc else "NOT included"}  {d}  ({what})')

In [ ]:
# @title Step 4 — ONLY if you have an older "-cleanup" folder: verify it
# WHAT: If you used the older analyze/apply steps, your old backups were moved to
#       <folder>-cleanup/superseded. This re-reads everything and proves no message
#       would be lost by deleting that "superseded" folder. (~1 hour per 125 GB)
# DO:   Run this cell. If you don't have a -cleanup folder it just says "Nothing to do".
# NEXT: If it ends with VERIFIED: in drive.google.com delete <folder>-cleanup/superseded,
#       then Trash -> Empty trash (frees most of the space). Then go to Step 5.
#       If it says NOT VERIFIED: delete nothing, and share the summary for help.
sup = os.path.join(OLD_WORK_DIR, 'superseded')
if os.path.isdir(sup) and any(f.lower().endswith('.xml') for f in os.listdir(sup)):
    !python -m smscleanup verify "$BACKUP_DIR" --work-dir "$OLD_WORK_DIR"
else:
    print('Nothing to do here (no -cleanup/superseded folder with backups). Go to Step 5.')

In [ ]:
# @title Step 5 — Merge every backup into ONE file
# WHAT: Reads every backup listed in Step 3 -- all folders, .zip files included --
#       (newest first) and writes each message once to
#       <folder>-merged/sms-<date>-merged.xml (plus calls-<date>-merged.xml if you
#       have call logs). Your original backups are NOT changed. Where a message is in
#       many backups, the copy from the newest backup is kept.
# NEEDS: about 5 GB of free Drive space.
# TIME: about 1 hour per 125 GB of backups. Keep this browser tab open. If Colab
#       disconnects, run Steps 1-3 again and then this step (it starts over; a
#       half-written file is never left behind).
# NEXT: It must end with "MERGE OK" (each type also says "self-check passed").
#       Then go to Step 6. If it says MERGE INCOMPLETE, read the PROBLEM lines.
!python -m smscleanup merge $INPUT_ARGS --out-dir "$MERGED_DIR" --tmp-dir /content/tmp

In [ ]:
# @title Step 6 — Verify the merged file, then gather the originals for deletion
# WHAT: Independently re-reads every original backup and checks each message is in
#       the merged file. Only if the result is VERIFIED are the originals moved into
#       <folder>-to-delete. If anything is MISSING, nothing is moved and the missing
#       messages are saved to <folder>-merged/verify-missing-*.xml.
#       (If one file can't be read because of Drive errors, it stays where it is and
#       the files that were fully verified are still moved.)
# TIME: about the same as Step 5.
# NEXT: If it ends with VERIFIED, go to Step 7. If NOT VERIFIED, delete nothing.
!python -m smscleanup verify-merged $INPUT_ARGS --out-dir "$MERGED_DIR" --move-sources-to "$TO_DELETE_DIR"

## Step 7 — Free the space (in drive.google.com, only after Step 6 says VERIFIED)
1. Delete the **`…-to-delete`** folder, and the old **`…-cleanup`** folder if you have one.
2. Open **Trash → Empty trash**. Storage is only released after this (the storage meter can take a few hours to update).
3. **Keep the `…-merged` folder.** Its `sms-…-merged.xml` (and `calls-…-merged.xml`) holds every message from every backup. To restore, point the backup app's restore at that file.
4. **On the phone:** set the backup app to incremental backups or "keep last N backups", so the old full backups don't pile up again.

Keep the merged file outside the app's own backup folder so the app's "delete old backups" setting can never remove it.